<h1>Boeing Final Assembly Rollout & Parking Optimization</h1>

<h2>User-Defined Values</h2>
<h3>1. Initialize for local machine</h3>

In [1]:
# user def / filepaths

user = 'jag'

userpaths = {
    'jag': '/Users/jishnughosh/Documents/GitHub/BSC-FGI_Scheduler',
    'ksg': '',
    'mpg': '/Users/mpgen/OneDrive - purdue.edu/PURDUE/Spring 26/IE 431 - Personal',
    'da': '',
    'lah': ''
}

rootpath = userpaths[user] # overwrite if necessary

filepaths = {
    'FAstatus': '/input/FA_Status_FGI_Handoff.xlsx',
    'FGI_Locations': '/input/FGI_Locations_wPriority.xlsx',
    'Nodes': '/input/Nodes.xlsx',
    'Move Times': '/input/adjusted_location_move_times.xlsx',
    'Paint Schedule': '/input/paint_schedules.xlsx'
}

<h3>2. Assumed Values</h3>
<p><strong>Changeable parameters:</strong></p>
<ul>
    <li><code>FGI_STAFFING_BYSHIFT</code>: Number of employees on each team for each shift</li>
    <li><code>FGI_CPJ</code>: Cost per job for each FGI team (# manhours required to complete 1 BTG task</li>
    <li><code>CENTERLINES</code>: Planes that must be moved for a move to occur starting at each location</li>

In [2]:
# algorithm toggles

FGI_STAFFING_BYSHIFT = { # FGI team: # manhours / 8hr shift
            1: {'structure': 16, 'systems': 60, 'declam': 16, 'test': 18},
            2: {'structure': 6, 'systems': 14, 'declam': 4, 'test': 0},
            3: {'structure': 0, 'systems': 0, 'declam': 0, 'test': 0}
        }

FGI_CPJ = { # FGI team: # manhours consumed / 1 BTG completed
        'structure': 6,
        'systems': 3.5,
        'declam': 6,
        'test': 8
    }

CENTERLINES = {
    'A1': None,
    'A2': None,
    'A3': None,
    'A4': None,
    'A5': None,
    'A6': None,
    'A7': None,
    'A8': None,
    'A9': ['C1'],
    'A10': ['C1', 'C2'],
    'BSC1': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'BSC2': ['C1', 'C2', 'C3', 'C3.5', 'C4', 'C5'],
    'C1': None,
    'C2': ['C1'],
    'C3': ['C1', 'C2'],
    'C3.5': ['C1', 'C2', 'C3'],
    'C4': ['C1', 'C2', 'C3', 'C3.5'],
    'C5': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'CR1': ['CR3', 'CR2'],
    'CR2': ['CR3'],
    'CR3': None,
    'D1': None,
    'D2': None,
    'F1': ['C1', 'C2'],
    'F2': ['C1', 'C2'],
    'L4': None,
    'L5': ['L4'],
    'Spur': None,
    'T1': None,
    'T2': None,
    'T3': None,
    'T4': None
}

FGI_TEAMS = ['structure', 'systems', 'declam', 'test']
BTG_TYPES = ['tot', 'p0', 'p1', 'p2', 'p3', 'engines', 'doors', 'test']

BTG_TYPES_BY_LABOR = { # relationships ONLY, NOT 1-to-1 btg conversion
    'structure': ['tot'],
    'systems': ['p2'],
    'declam': ['p3', 'engines'],
    'test': ['test']
}
FGI_TEAMS_BY_BTG_TYPE = { # relationships ONLY, NOT 1-to-1 btg conversion
    'tot': ['structure', 'systems', 'declam', 'test'],
    'FGI_tot': ['structure'],
    'p2': ['systems'],
    'p3': ['declam'],
    'engines': ['declam'],
    'test': ['test']
}
for type in BTG_TYPES:
    if type not in FGI_TEAMS_BY_BTG_TYPE.keys():
        FGI_TEAMS_BY_BTG_TYPE[type] = None


<p>|\==><i>NOTE: THESE PARAMETERS WILL BE OVERWRITTEN IF INPUT_TYPE IS SET TO FGI_LIVESTATE</p>

<h2>Initialization</h2>
<h3>1. Import Libraries</h3>
<p><strong>Required Installations:</strong></p>
<ul>
    <li>pandas</li>
    <li>numpy</li>
    <li>datetime</li>
    <li>matplotlib.pyplot</li>
    <li>seaborn</li>
    <li>math</li>
</ul>

In [3]:
import pandas as pd
import numpy as np
import datetime as dt
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import math
from math import dist
# import ast
# import heapq
# import re
import os

<h3>2. Run Conditions</h3>
<p>These parameters set the conditions under which the algorithm will generate the schedule. Be sure that this is updated on each run of the main placement algorithm</p>

In [4]:
# a) planning horizon
STARTDATE='2024-06-10'
ENDDATE='2026-03-10'
FORECAST_UNTIL_COMPLETION = True

# b) location parameters

NEW_FA_ONLINE = False
DISCONTINUED_LOCATIONS = ['S1', 'S2', 'S3', 'Spur']
INCLUDE_FA_LOCATIONS = False

# c) schedule parameters
PLANNED_BUFFER_DAYS = []
IMPORT_PAINT_SCHEDULE = False


# d) input parameters
input_types = [
    'FARO_SCORECARD_WITH_MILESTONES',
    'FGI_LIVESTATE'
]
INPUT_TYPE = input_types[1]

# e) output/export parameters
EXPORT_TO_EXCEL = True
EXPORT_PATH = '/output/'
CODECELL_OUTPUT = True
EXPORT_TO_FGI_LIVESTATE = True


<p><i>|==> Initialize based on run conditions </i></p>

<h3>AP, Location, and Labor Import</h3>

In [5]:
# dataframe builder functions
def merge_APdata(faro_scorecard, p3_milestones, tankClosure):
    ## organize milestone/tankClosure by LN
    tank_lookup = (
        tankClosure[['LINE_NUMBER', 'TankStatus']]
        .drop_duplicates(subset='LINE_NUMBER')
        .rename(columns={'LINE_NUMBER': 'LN'})
    )
    p3_lookup = (
        p3_milestones
        .rename(columns={'P': 'LN'})
        .copy()
    )

    faro_scorecard = faro_scorecard.copy()
    faro_scorecard['LN'] = pd.to_numeric(faro_scorecard['LN'], errors='coerce')

    tank_lookup['LN'] = pd.to_numeric(tank_lookup['LN'], errors='coerce')
    p3_lookup['LN'] = pd.to_numeric(p3_lookup['LN'], errors='coerce')

    ## store LN as str
    faro_scorecard['LN'] = faro_scorecard['LN'].astype(str)
    tank_lookup['LN'] = tank_lookup['LN'].astype(str)
    p3_lookup['LN'] = p3_lookup['LN'].astype(str)

    ## merge all dataframes
    ap_df = (
        faro_scorecard
        .merge(tank_lookup, on='LN', how='left')
        .merge(p3_lookup, on='LN', how='left')
    )

    ## merge all dataframes
    ap_df = (
        faro_scorecard
        .merge(tank_lookup, on='LN', how='left')
        .merge(p3_lookup, on='LN', how='left')
    )

    return ap_df

def build_ap_df(faro_scorecard, p3_milestones, tankClosure):
    ap_data = merge_APdata(faro_scorecard, p3_milestones, tankClosure)

    rows = []

    for _, row in ap_data.iterrows():
        ln = int(row['LN'])

        rows.append({
            'LN': ln,
            'FA RO': row['FA RO'],
            'FA RO to B1R': row['FA RO to B1R'],
            'Total Counters': row['Total Counters'],
            'TankStatus': row['TankStatus'],
            'Ceilings': row['Ceilings'],
            'Initial Tests Run': row['Initial Tests Run'],

            # BTG attributes
            'BTG_tot': row['Total BTG'],
            'BTG_p0': row['P0 BTG'],
            'BTG_p1': row['P1 BTG'],
            'BTG_p2': row['P2 BTG'],
            'BTG_p3': row['P3 BTG'],
            'BTG_engines': row['Engines BTG'],
            'BTG_doors': row['Doors BTG'],
            'BTG_test': row['Test BTG'],

            # P3 milestone attributes
            'P3_Engine Hang': row['Engine Hang'],
            'P3_Flight Controls': row['Flight Controls'],
            'P3_Gear Swing': row['Gear Swing'],
            'P3_Medium Pressure Test': row['Medium Pressure Test'],
            'P3_Oil On': row['Oil On'],
            'P3_Power On': row['Power On'],
            'P3_Engine_Type': row['Engine_Type'],
            'P3_Milestone_Completion_Percentage': row['Milestone_Completion_Percentage'],

            # Shake attributes
            'shakes_complete': row['shakes_completed'],
            'shakes_req': row['shakes_req']
        })

    return pd.DataFrame(rows)

def build_location_df(fa_priority, centerlines):
    rows = []
    seen_locations = set()

    for _, row in fa_priority.iterrows():
        loc = row['Location']
        seen_locations.add(loc)
        centerline_deps = centerlines.get(loc, None)
        

        rows.append({
            'Location': loc,
            'Future State Priority': row['Future State Priority'],
            'Date Online': row['Date Online'],
            'Owner': row['Owner'],
            'tooling_jacking': row['Jacking'] == 'Y',
            'tooling_wings': row['Wings'] == 'Y',
            'tooling_tankClosure': row['Tank Closure'] == 'Y',
            'centerline_dependencies': None if centerline_deps is None else ', '.join(centerline_deps),
            # 'obstructions': None,
            # 'notes': None
        })

    return pd.DataFrame(rows)

def build_labor_df(fgi_staffing_byshift, fgi_cpj, startdate, enddate):
    rows = []

    # FGI staffing by shift
    for shift, teams in fgi_staffing_byshift.items():
        for team, manhours in teams.items():
            rows.append({
                'category': 'FGI_STAFFING_BYSHIFT',
                'shift': shift,
                'team': team,
                'value': manhours
            })

    # FGI CPJ
    for team, cpj in fgi_cpj.items():
        rows.append({
            'category': 'FGI_CPJ',
            'shift': None,
            'team': team,
            'value': cpj
        })

    # Dates
    rows.append({
        'category': 'CONFIG',
        'shift': None,
        'team': 'STARTDATE',
        'value': startdate
    })
    rows.append({
        'category': 'CONFIG',
        'shift': None,
        'team': 'ENDDATE',
        'value': enddate
    })

    return pd.DataFrame(rows)


# build dataframes from FARO scorecare with milestones
if INPUT_TYPE == 'FARO_SCORECARD_WITH_MILESTONES':
    def clean_FAstatus(faro_scorecard, tankClosure, p3_milestones):
        ## FARO Scorecard Cleaning
        # only take rows with valid LNs
        faro_scorecard = faro_scorecard[pd.to_numeric(faro_scorecard['LN'], errors='coerce').notna()].copy()
        # remove unnamed columns
        faro_scorecard = faro_scorecard.loc[:, ~faro_scorecard.columns.astype(str).str.contains(r'^Unnamed')]

        # split zone shakes column into two columns: completed/required
        faro_scorecard[['shakes_completed', 'shakes_req']] = faro_scorecard['Zone Shakes'].astype(str).str.split('/', expand=True)
        # only take 
        faro_scorecard['p3_milestones'] = faro_scorecard['P3 Milestones'].astype(str).str.split('/').str[0]

        # set datatypes
        faro_scorecard = (
            faro_scorecard.assign(
                LN=lambda df: pd.to_numeric(df['LN'], errors='coerce').astype(int),
                **{
                    'FA RO to B1R': lambda df: pd.to_numeric(df['FA RO to B1R'], errors='coerce'),
                    'Total Counters': lambda df: pd.to_numeric(df['Total Counters'], errors='coerce').fillna(0),
                    'Total BTG': lambda df: pd.to_numeric(df['Total BTG'], errors='coerce').fillna(0),
                    'P0 BTG': lambda df: pd.to_numeric(df['P0 BTG'], errors='coerce').fillna(0),
                    'P1 BTG': lambda df: pd.to_numeric(df['P1 BTG'], errors='coerce').fillna(0),
                    'P2 BTG': lambda df: pd.to_numeric(df['P2 BTG'], errors='coerce').fillna(0),
                    'P3 BTG': lambda df: pd.to_numeric(df['P3 BTG'], errors='coerce').fillna(0),
                    'Engines BTG': lambda df: pd.to_numeric(df['Engines BTG'], errors='coerce').fillna(0),
                    'Doors BTG': lambda df: pd.to_numeric(df['Doors BTG'], errors='coerce').fillna(0),
                    'Test BTG': lambda df: pd.to_numeric(df['Test BTG'], errors='coerce').fillna(0),
                    'Tank Closure': lambda df: df['Tank Closure'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int),
                    'Ceilings': lambda df: pd.to_numeric(df['Ceilings'], errors='coerce').fillna(0),
                    'Initial Tests Run': lambda df: (
                        df['Initial Tests Run'].astype(str)
                        .str.replace('%', '', regex=False)
                        .replace('nan', 0)
                        .replace('', 0)
                        .astype(float) / 100
                    ),
                    'shakes_completed': lambda df: pd.to_numeric(df['shakes_completed'], errors='coerce').fillna(0).astype(int),
                    'shake_req': lambda df: pd.to_numeric(df['shakes_req'], errors='coerce').fillna(0).astype(int),
                    'p3_milestones': lambda df: pd.to_numeric(df['p3_milestones'], errors='coerce').fillna(0).astype(int),
                }
            )
        )

        ## TANK CLOSURE Cleaning
        tankClosure['LINE_NUMBER'] = pd.to_numeric(tankClosure['LINE_NUMBER'], errors='coerce')
        tankClosure['Complete_Jobs'] = pd.to_numeric(tankClosure['Complete_Jobs'], errors='coerce').fillna(0)
        tankClosure['Total_Jobs'] = pd.to_numeric(tankClosure['Total_Jobs'], errors='coerce').fillna(0)
        tankClosure['TankStatus'] = tankClosure['TankStatus'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int)

        ## P3 Milestone Cleaning
        p3_milestones['Engine_Type'] = p3_milestones['Milestone'].astype(str).str.extract(r'\((.*?)\)')
        p3_milestones['Milestone'] = p3_milestones['Milestone'].astype(str).str.replace(r'\s*\(.*?\)', '', regex=True)
        p3_milestones = (
            p3_milestones.pivot_table(index='P', columns='Milestone', values='Completed_Jobs', aggfunc='sum')
            .assign(
                Engine_Type=p3_milestones.groupby('P')['Engine_Type'].first(),
                Milestone_Completion_Percentage=p3_milestones.groupby('P')['STATUS (1 Complete, 0 Still Open)'].mean()
            )
            .fillna(-1)
            .reset_index()
        )

        return faro_scorecard, tankClosure, p3_milestones

    def clean_nodeData(nodes, node_adj):
        ## clean nodes
        nodes = (
            nodes
            .drop(columns=[c for c in nodes.columns if str(c).startswith('Unnamed')], errors='ignore')
            .dropna(how='all')
            .assign(
                node_id=lambda df: df['node_id'].astype('string').str.strip(),
                x=lambda df: pd.to_numeric(df['x'], errors='coerce'),
                y=lambda df: pd.to_numeric(df['y'], errors='coerce'),
                type=lambda df: df['type'].astype('string').str.strip(),
                req_centerline=lambda df: df['req_centerline'].astype('string').str.strip()
            )
            .replace({'': pd.NA})
            .dropna(subset=['node_id', 'x', 'y'])
            .reset_index(drop=True)
        )

        ## clean node_adj 
        node_adj = (
            node_adj
            .drop(columns=[c for c in node_adj.columns if str(c).startswith('Unnamed')], errors='ignore')
            .dropna(how='all')
            .assign(
                from_node=lambda df: df['from_node'].astype('string').str.strip(),
                to_node=lambda df: df['to_node'].astype('string').str.strip()
            )
            .replace({'': pd.NA})
            .dropna(subset=['from_node', 'to_node'])
            .drop_duplicates()
            .reset_index(drop=True)
        )

        return nodes, node_adj

    def import_data(rootpath, filepaths = {'FAstatus': 'FA_Status_FGI_Handoff.xlsx','FGI_Locations': 'FGI_Locations_wPriority.xlsx','Nodes': 'Nodes.xlsx'}):
        
        ## FA STATUS DATA IMPORT
        path = rootpath + filepaths['FAstatus']
        # read FARO scorecard as df
        faro_scorecard = pd.read_excel(path, sheet_name='FARO_Scorecard',header=2,engine='openpyxl')
        tankClosure = pd.read_excel(path,sheet_name='Tank_Closure_Detail',engine='openpyxl')
        p3_milestones = pd.read_excel(path, sheet_name='P3 Milestone Detail',engine='openpyxl')

        ## FGI LOCATION DATA IMPORT
        path = rootpath + filepaths['FGI_Locations']
        fa_priority = pd.read_excel(path, sheet_name='FA Priority', header=1,engine='openpyxl')

        ## NODES DATA IMPORT
        path = rootpath + filepaths['Nodes']
        nodes = pd.read_excel(path,sheet_name='Node',engine='openpyxl')
        node_adj = pd.read_excel(path, sheet_name='adjacency',engine='openpyxl')

        ## DATA CLEANING
        faro_scorecard, tankClosure, p3_milestones = clean_FAstatus(faro_scorecard, tankClosure, p3_milestones)

        fa_priority = (
            fa_priority
            .drop(columns=[c for c in fa_priority.columns if str(c).startswith('Unnamed')], errors='ignore')
            .dropna(how='all')
            .reset_index(drop=True)
        )

        nodes, node_adj = clean_nodeData(nodes, node_adj)

        return faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj

    faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj = import_data(rootpath, filepaths)
    ap_df = build_ap_df(faro_scorecard, p3_milestones, tankClosure)

    location_df = build_location_df(
        fa_priority=fa_priority,
        centerlines=CENTERLINES
    )

    labor_df = build_labor_df(
        fgi_staffing_byshift=FGI_STAFFING_BYSHIFT,
        fgi_cpj=FGI_CPJ,
        startdate=STARTDATE,
        enddate=ENDDATE
    )


    staffing_df = labor_df[labor_df['category'] == 'FGI_STAFFING_BYSHIFT'].copy()
    staffing_df['shift'] = staffing_df['shift'].astype(int)
    staffing_df['value'] = pd.to_numeric(staffing_df['value'])
    FGI_STAFFING_BYSHIFT = {
        shift: skill.set_index('team')['value'].to_dict()
        for shift, skill in staffing_df.groupby('shift', sort=True)
    }

    FGI_CPJ = (
        labor_df[labor_df['category'] == 'FGI_CPJ']
        .assign(value=lambda df: pd.to_numeric(df['value']))
        .set_index('team')['value']
        .to_dict()
    )

# build dataframes from FGI_LIVESTATE
if INPUT_TYPE == 'FGI_LIVESTATE':

    def load_live_state(path='FGI_liveState.xlsx'):
        required_columns = {
            'ap_data': [
                'LN', 'FA RO', 'FA RO to B1R', 'Total Counters', 'TankStatus', 'Ceilings',
                'Initial Tests Run', 'BTG_tot', 'BTG_p0', 'BTG_p1', 'BTG_p2', 'BTG_p3',
                'BTG_engines', 'BTG_doors', 'BTG_test', 'P3_Engine Hang',
                'P3_Flight Controls', 'P3_Gear Swing', 'P3_Medium Pressure Test',
                'P3_Oil On', 'P3_Power On', 'P3_Engine_Type',
                'P3_Milestone_Completion_Percentage', 'shakes_complete', 'shakes_req'
            ],
            'location_data': [
                'Location', 'Future State Priority', 'Date Online', 'Owner',
                'tooling_jacking', 'tooling_wings', 'tooling_tankClosure',
                'centerline_dependencies', 'obstructions', 'notes'
            ],
            'labor_data': ['category', 'shift', 'team', 'value']
        }

        numeric_ap_columns = [
            'LN', 'FA RO to B1R', 'Total Counters', 'TankStatus', 'Ceilings',
            'Initial Tests Run', 'BTG_tot', 'BTG_p0', 'BTG_p1', 'BTG_p2', 'BTG_p3',
            'BTG_engines', 'BTG_doors', 'BTG_test', 'P3_Engine Hang',
            'P3_Flight Controls', 'P3_Gear Swing', 'P3_Medium Pressure Test',
            'P3_Oil On', 'P3_Power On', 'P3_Milestone_Completion_Percentage',
            'shakes_complete', 'shakes_req'
        ]

        tooling_columns = ['tooling_jacking', 'tooling_wings', 'tooling_tankClosure']

        def _normalize_bool(value):
            if pd.isna(value):
                return False
            if isinstance(value, (bool, np.bool_)):
                return bool(value)
            if isinstance(value, (int, float)):
                return value != 0

            value_str = str(value).strip().lower()
            if value_str in {'true', 't', 'yes', 'y', '1'}:
                return True
            if value_str in {'false', 'f', 'no', 'n', '0', ''}:
                return False

            raise ValueError(f'Invalid boolean value in location_data: {value}')

        if not os.path.exists(path):
            raise ValueError(f'Live state workbook not found: {path}')

        try:
            workbook = pd.ExcelFile(path, engine='openpyxl')
        except FileNotFoundError as exc:
            raise ValueError(f'Live state workbook not found: {path}') from exc
        except Exception as exc:
            raise ValueError(f'Unable to open live state workbook: {path}') from exc

        missing_sheets = [sheet for sheet in required_columns if sheet not in workbook.sheet_names]
        if missing_sheets:
            raise ValueError(
                f"Missing required sheet(s) in {path}: {', '.join(missing_sheets)}"
            )

        ap_df = pd.read_excel(workbook, sheet_name='ap_data')
        location_df = pd.read_excel(workbook, sheet_name='location_data')
        labor_df = pd.read_excel(workbook, sheet_name='labor_data')

        frames = {
            'ap_data': ap_df,
            'location_data': location_df,
            'labor_data': labor_df
        }

        for sheet_name, expected_columns in required_columns.items():
            missing_columns = [col for col in expected_columns if col not in frames[sheet_name].columns]
            if missing_columns:
                raise ValueError(
                    f"Missing required column(s) in sheet '{sheet_name}': {', '.join(missing_columns)}"
                )

        ap_df = ap_df.copy()
        ap_df['FA RO'] = pd.to_datetime(ap_df['FA RO'], errors='coerce')
        for column in numeric_ap_columns:
            ap_df[column] = pd.to_numeric(ap_df[column], errors='coerce')

        if ap_df['LN'].notna().all():
            ap_df['LN'] = ap_df['LN'].astype('Int64')

        location_df = location_df.copy()
        location_df['Location'] = location_df['Location'].astype('string').str.strip()
        location_df = location_df[location_df['Location'].notna() & location_df['Location'].ne('')].reset_index(drop=True)
        for column in tooling_columns:
            location_df[column] = location_df[column].map(_normalize_bool).astype(bool)

        labor_df = labor_df.copy()
        labor_df['shift'] = pd.to_numeric(labor_df['shift'], errors='coerce').astype('Int64')

        return ap_df, location_df, labor_df
    
    ap_df, location_df, labor_df = load_live_state(path='FGI_liveState.xlsx')
    
if EXPORT_TO_FGI_LIVESTATE:

    output_path = rootpath + EXPORT_PATH + 'FGI_liveState.xlsx'

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        ap_df.to_excel(writer, sheet_name='ap_data', index=False)
        location_df.to_excel(writer, sheet_name='location_data', index=False)
        labor_df.to_excel(writer, sheet_name='labor_data', index=False)

    if CODECELL_OUTPUT: print(output_path)

if CODECELL_OUTPUT:
    def line(char='—', length = 100):
        line = char
        while len(line) < length:
            line += char
        print(line+'\n')
    # HEADER_NUMBER = 0
    def header(text, char='_', length=100):
        # HEADER_NUMBER = HEADER_NUMBER + 1
        padding = [f'{char}| ', f' |{char}']
        line = f'{char}| {text} |{char}'
        while len(line) < length:
            padding = [char+padding[0], padding[1]+char]
            line = padding[0] + text + padding[1]

        print(line)
        

/Users/jishnughosh/Documents/GitHub/BSC-FGI_Scheduler/output/FGI_liveState.xlsx


<h3>Move Time Import</h3>

In [6]:
def load_move_times(filepath='input/adjusted_location_move_times.xlsx'):
    df = pd.read_excel(filepath, index_col=0)
    # set location names to strings
    df.index = df.index.astype(str)
    df.columns = df.columns.astype(str)

    df = df.apply(pd.to_numeric, errors='coerce')
    move_times = df.to_dict(orient='index')
    return move_times

def add_move_times(fgi, move_times):
    # ensure all locations exist
    for from_loc, to_dict in move_times.items():
        if from_loc not in fgi.Locations:
            fgi.add_Location(
                from_loc,
                Location(priority=0, dateOnline='Now', name=from_loc)
            )

        for to_loc in to_dict:
            if to_loc not in fgi.Locations:
                fgi.add_Location(
                    to_loc,
                    Location(priority=0, dateOnline='Now', name=to_loc)
                )

    # assign move times directly
    for from_loc, to_dict in move_times.items():
        loc_obj = fgi.Locations[from_loc]
        for to_loc, time in to_dict.items():
            loc_obj.set_time_to(to_loc, time)

    return fgi

In [7]:
move_times = load_move_times()

In [8]:
def load_paint_schedule(filepath=filepaths['Paint Schedule'], sheet_name='Historical'):

    paint_df = pd.read_excel(filepath, sheet_name=sheet_name, header=2, engine='openpyxl')
    paint_df = paint_df[['Date', 'BSC1', 'BSC2']].copy()

    paint_df['Date'] = pd.to_datetime(paint_df['Date'], errors='coerce').dt.normalize()
    paint_df = paint_df[paint_df['Date'].notna()].reset_index(drop=True)

    paint_schedule = {}

    for _, row in paint_df.iterrows():
        date = row['Date']
        paint_schedule[date] = {}

        for bay in ['BSC1', 'BSC2']:
            if pd.isna(row[bay]):
                paint_schedule[date][bay] = None
            else:
                paint_schedule[date][bay] = str(int(row[bay]))

    return paint_schedule


<p>|\<i>==> Now, dataframes have been built to describe APs, Locations, and Labor Allocation pararameters to build the schedule</i></p>

<h2>Scheduler</h2>
<h3>1. Class Initialization</h3>
<h4>a) Definition</h4>

In [9]:
class AP:
    ## ————————————OBJECT INITIALIZATION——————————————————————————————————————————————————————————————————————————
    ## Initialize AP with FARO attributes
    def __init__(self,LN,faro,toB1R,counters,btg=None,tankClosed=False,ceilings=0,testsRun=0,p3_milestones=None,shakes=None):
        # FA attributes at RO
        self.LN=LN
        self.faro=faro
        self.toB1R=toB1R
        self.counters=counters
        self.btg=btg if btg is not None else {"tot":0,"p0":0,"p1":0,"p2":0,"p3":0,"engines":0,"doors":0,"test":0}
        self.tankClosed=tankClosed
        self.ceilings=ceilings
        self.testsRun=testsRun
        self.p3milestones=p3_milestones if p3_milestones is not None else {'tot':6,'complete':0}
        self.shakes = shakes if shakes is not None else {'complete': 0, 'req': 4}
        
        # scheduling attributes
        self.schedule={}
        self.moveReq=False
        self.path=[]

        # state attributes
        self.Location = None
        self.taskState = 'FA'
        self.destination = None


        ##_ _ _ _ _ _ _ _ _ _ _ _| convert FARO BTG <=> FGI BTG |_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
        self.FGI_btg = {'FGI_tot': self.btg['tot'] - self.btg['p0'] - self.btg['p1'], 
                        'structure': math.ceil(0.1 * (self.btg['tot'] - self.btg['p0'] - self.btg['p1'])), 
                        'systems': math.ceil(self.btg['p2']),
                        'declam': math.ceil(self.btg['p3'] + self.btg['engines']),
                        'test': self.btg['test']
                        }
        self.taskCompletion = {
            'move': self.moveReq,
            'compass': False,
            'paint': False,
            'structure': False if self.FGI_btg['structure'] > 0 else True,
            'systems': False if self.FGI_btg['systems'] > 0 else True,
            'declam': False if self.FGI_btg['declam'] > 0 else True,
            'test': False if self.FGI_btg['test'] > 0 else True
        }

    ## ————————————| GET METHODS |——————————————————————————————————————————————————————————————————————————
    # clean attributes into normalized datatypes for use within algorithm
    def get_LN(self): return str(self.LN) # —
    def get_FAROdate(self): return pd.to_datetime(self.faro)
    def get_days_toB1R(self): return pd.Timedelta(days=self.toB1R)
    def get_counters(self): return int(self.counters)
    def get_BTG(self, category): return pd.to_numeric(self.btg[category]) if category in self.btg.keys() else False
    def get_percent_ceilings(self): return pd.to_numeric(self.ceilings)
    def get_percent_testsRun(self): return pd.to_numeric(self.testsRun)
    def get_daystoB1R(self): return pd.Timedelta(days=self.toB1R)
    def get_fgi_btg(self, team): return pd.to_numeric(self.FGI_btg[team]) if team in self.FGI_btg.keys() else False

    
    ## ————————————| SCHEDULING METHODS |——————————————————————————————————————————————————————————————————————————
    
    def set_taskState(self, state):
        AP_TASK_STATES = ['FA', 'RO', 'paint', 'compass', 'shake', 'btg_completion', 'tankClosure', 'toDC', 'exit', 'delivered']
        if state in AP_TASK_STATES:
            self.taskState = state
        else:
            if CODECELL_OUTPUT:
                header('INVALID METHOD CALL')
                print(f'LN: {self.LN} taskState set to {state}')
                line()
            return False

    def get_move_candidates(self, fgi, allow_temp_locations=False):
        origin = self.Location
        candidates = []

        for loc_name, loc in fgi.Locations.items():
            if origin is not None and loc.name == origin.name:
                continue

            if loc.is_temp and not allow_temp_locations:
                continue

            if not loc.canPlace(self):
                continue

            if origin is None:
                move_time = 0
            else:
                move_time = origin.time_to.get(loc_name, np.inf)

                if not np.isfinite(move_time):
                    continue

            candidates.append({
                'destination': loc,
                'destination_name': loc.name,
                'priority': loc.priority,
                'move_time': move_time
            })

        return sorted(candidates, key=lambda c: (c['priority'], c['move_time']))
    
    def isMoveReq(self): return self.moveReq

    def requireMove(self, destination=None): 
        self.moveReq = True
        self.destination = destination
        if CODECELL_OUTPUT:
            header('MOVE REQUIRED')
            current_loc = self.Location.name if self.Location is not None else None
            print(f'LN: {self.LN} requires move from {current_loc}\nTask State: {self.taskState}')
            line()

    def complete_BTG(self, category, btg_count=0, byLabor=False):
        ## input checking
        if btg_count is None or btg_count <= 0:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: complete_BTG')
                print('tried to complete btg on an AP when btg_count was 0 or no parameter given')
                print(f'LN: {self.LN} category: {category} btg_count: {btg_count}')
                line()
            return 0

        if not byLabor and category not in self.btg:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: complete_BTG')
                print(f'invalid BTG category when byLabor=False: {category}')
                print(f'valid categories: {list(self.btg.keys())}')
                line()
            return btg_count

        if byLabor and category not in self.FGI_btg:
            if CODECELL_OUTPUT:
                header('METHOD ERROR: complete_BTG')
                print(f'invalid labor category when byLabor=True: {category}')
                print(f'valid categories: {list(self.FGI_btg.keys())}')
                line()
            return btg_count

        remaining_btg = btg_count

        ## complete BTG directly by BTG category
        if not byLabor:
            available = max(self.btg[category], 0)
            completed = min(available, remaining_btg)

            self.btg[category] -= completed
            self.btg['tot'] -= completed

            if category in FGI_TEAMS_BY_BTG_TYPE and FGI_TEAMS_BY_BTG_TYPE[category] is not None:
                for group in FGI_TEAMS_BY_BTG_TYPE[category]:
                    if group in self.FGI_btg:
                        self.FGI_btg[group] = max(self.FGI_btg[group] - completed, 0)

            remaining_btg -= completed

        ## complete BTG by labor bucket
        else:
            available = max(self.FGI_btg[category], 0)
            completed = min(available, remaining_btg)

            self.FGI_btg[category] -= completed

            if category in BTG_TYPES_BY_LABOR:
                for btg_type in BTG_TYPES_BY_LABOR[category]:
                    if btg_type in self.btg:
                        self.btg[btg_type] = max(self.btg[btg_type] - completed, 0)

            self.btg['tot'] = max(self.btg['tot'] - completed, 0)
            remaining_btg -= completed

        return True, remaining_btg

class Location:
    def __init__(self,priority,dateOnline,name,owner=None,tooling=None,centerlines=None):
        self.priority=priority

        if dateOnline == 'Now':
            self.isOnline = True
        elif dateOnline == 'At R10' and NEW_FA_ONLINE == True:
            self.isOnline = False
        else:
            self.isOnline = False
        
        self.name=name
        self.is_temp = str(self.name).startswith('N')
        self.owner=owner
        self.tooling=tooling if tooling is not None else {'jacking':False,'wings':False,'tankClosure':False}
        # self.obstructions=obstructions if obstructions is not None else []
        # self.notes=notes if notes is not None else []
        self.centerlines = [
            centerline.strip()
            for centerline in str(centerlines).split(',')
            if centerline.strip() not in ['', 'None', 'nan', 'N/A']
        ]
            
        self.schedule={}
        self.time_to = {}
        self.AP = None

    def canUse(self,tool): 
        if tool in self.tooling.keys():
            return self.tooling[tool]
        
    def isAvailable(self): return True if self.AP == None else False

    def canPlace(self, ap):
        if self.isOnline:
            return self.isAvailable()
        elif self.AP == ap:
            return False
        else:
            return False
    
    def assign(self, AP, date=None):
        if self.AP is not None:
            header('ERROR', '~')
            print("AP assigned to unavailable location")
            line('~')
            return False
        
        self.AP = AP 
        if date is not None:
            self.schedule[date] = AP.get_LN()
        return True
    def unassign(self,date=None):
        if CODECELL_OUTPUT:
            header('MOVE', ' - ')
            print(f'{self.AP.get_LN()} removed from location {self.name}')
            line(' - ')

        self.AP = None
        


    def obstructs(self,obstruction): return obstruction in self.obstructions
    def isHigherPriorityThan(self,other): 
        return self.priority < other.priority
    def clear_schedule(self): self.schedule={}
    def set_time_to(self, other, move_time):
        self.time_to[other] = move_time

# =====================MAIN SCHEDULING CLASS===========================
## FGI manages all APs currently owned by FGI and all active locations within FGI
## currently also manages queues for moves, fgi tasks, and labor

class FGI:
    def __init__(self, labor, CPJ):
        self.labor = labor
        self.CPJ = CPJ
        self.APs = {}
        self.Locations = {}
        # self.locations = self.Locations
        self.queues = {
            'move': [],
            'FGI task': {
                'compass': [],
                'paint': []
            },
            'labor': {
                'structure': [],
                'systems': [],
                'declam': [],
                'test': []
            }
        }
        self.schedule = {}
        self.chickenTracks = {}
        self.apStateRows = []
        self.shift = 0

    def get_queue(self, queue):
        if queue == 'all':
            return self.queues
        else:
            return self.queues[queue]

    def remove_from_queue(self, queue_group, LN, subqueue=None):
        if queue_group == 'move':
            while LN in self.queues['move']:
                self.queues['move'].remove(LN)
            return

        if subqueue is None:
            return

        queue = self.queues[queue_group][subqueue]

        while LN in queue:
            queue.remove(LN)

    def remove_from_all_queues(self, LN):
        self.remove_from_queue('move', LN)

        for task in self.queues['FGI task']:
            self.remove_from_queue('FGI task', LN, subqueue=task)

        for team in self.queues['labor']:
            self.remove_from_queue('labor', LN, subqueue=team)

    def is_in_any_queue(self, LN):
        if LN in self.queues['move']:
            return True

        for queue in self.queues['FGI task'].values():
            if LN in queue:
                return True

        for queue in self.queues['labor'].values():
            if LN in queue:
                return True

        return False

    def get_labor(self, team, shift=0):
        if self.shift == 0 and CODECELL_OUTPUT:
            header('ERROR', '~')
            print('Shift must be initialized for FGI object')
            line()
            return None
            

        if team == 'all':
            return self.labor[shift]
        else:
            return self.labor[shift][team]
        
    def get_btg_completion(self, team, shift=0):
        if shift == 0:
            header('ERROR', '~')
            return False
        else:
            return pd.to_numeric(self.labor[shift][team] / self.CPJ[team])
        
    def add_ap(self, ap):
        LN = ap.get_LN()

        self.APs[LN] = ap
        ap.set_taskState('RO')

        if LN not in self.queues['FGI task']['paint']:
            self.queues['FGI task']['paint'].append(LN)

        if LN not in self.queues['FGI task']['compass']:
            self.queues['FGI task']['compass'].append(LN)

        for team in self.queues['labor']:
            if not ap.taskCompletion.get(team, False):
                if LN not in self.queues['labor'][team]:
                    self.queues['labor'][team].append(LN)

        if CODECELL_OUTPUT:
            header('AP ADDED TO FGI')
            print(f'LN {LN} added to FGI')
            print(f"Paint queue length: {len(self.queues['FGI task']['paint'])}")
            print(f"Compass queue length: {len(self.queues['FGI task']['compass'])}")
            for team in self.queues['labor']:
                print(f"{team} queue length: {len(self.queues['labor'][team])}")
            line()

    def add_Location(self, name, location_obj):
        self.Locations[name] = location_obj
        self.chickenTracks[name] = None
        if CODECELL_OUTPUT:
            line()
            print(f'Location {name} added to FGI')
            print(f'Current locations in FGI: {list(self.Locations.keys())}')
            line()

    def assign_labor(self, team, max_btg_completion=30, date=None):
    
        btg_completed = 0
        worked_lns = []

        if team not in self.queues['labor']:
            raise ValueError(f'Invalid labor team: {team}')

        queue = self.queues['labor'][team]
        i = 0

        while i < len(queue) and btg_completed < max_btg_completion:
            LN = queue[i]
            if LN not in self.APs:
                queue.pop(i)
                continue

            ap = self.APs[LN]

            remaining_capacity = max_btg_completion - btg_completed
            success, remaining_btg = ap.complete_BTG(team, btg_count=remaining_capacity, byLabor=True)

            if success:
                done = remaining_capacity - remaining_btg
                btg_completed += done

                if done > 0:
                    worked_lns.append(LN)
                    if hasattr(self, 'trace'):
                        self.trace.record_btg(date, LN, team, done)

            ## if this queue's work is now complete, remove LN from queue
            if ap.get_fgi_btg(team) <= 0:
                self.queues['labor'][team].pop(i)
                ap.taskCompletion[team] = True
                continue
            else:
                i += 1

        return worked_lns, btg_completed
    
    def move_ap(self, LN, destination, date=None):
        moved, move_time = self.move_ap_direct(LN, destination, date=date, record_trace=True)

        if not moved:
            if CODECELL_OUTPUT:
                header('MOVE BLOCKED', '~')
                print(self.last_move_error)
                line('~')
            return False, 0

        AP = self.APs[LN]
        AP.moveReq = False
        AP.destination = None

        self.remove_from_queue('move', LN)

        if CODECELL_OUTPUT:
            header('MOVE EXECUTED')
            print(f'LN {LN}: {None if AP.path == [] else AP.path[-1]} -> {destination.name} | move_time={move_time}')
            line()

        return True, move_time

    def get_centerline_blockers(self, destination):
        blockers = []

        if destination.centerlines is None:
            return blockers

        for blocker_loc_name in destination.centerlines:
            blocker_loc_name = str(blocker_loc_name).strip()

            if blocker_loc_name not in self.Locations:
                continue

            blocker_loc = self.Locations[blocker_loc_name]

            if blocker_loc.AP is not None:
                blockers.append({
                    'loc_name': blocker_loc_name,
                    'loc': blocker_loc,
                    'ap': blocker_loc.AP,
                    'LN': blocker_loc.AP.get_LN()
                })

        return blockers

    def find_temp_location(self, origin):
        candidates = []

        for loc_name, loc in self.Locations.items():
            if not loc.is_temp:
                continue

            if not loc.isAvailable():
                continue

            if origin is None:
                move_time = 0
                return_time = 0
            else:
                move_time = origin.time_to.get(loc_name, np.inf)
                return_time = loc.time_to.get(origin.name, np.inf)

                if not np.isfinite(move_time) or not np.isfinite(return_time):
                    continue

            candidates.append((move_time, loc))

        if len(candidates) == 0:
            return None

        return sorted(candidates, key=lambda x: x[0])[0][1]

    def get_closest_normal_destination(self, AP, banned_locations=None, allow_banned_if_no_other=True):
        if banned_locations is None:
            banned_locations = []

        origin = AP.Location

        preferred = []
        fallback = []

        for loc_name, loc in self.Locations.items():
            if loc.is_temp:
                continue

            if origin is not None and loc_name == origin.name:
                continue

            if not loc.canPlace(AP):
                continue

            if origin is None:
                move_time = 0
            else:
                move_time = origin.time_to.get(loc_name, np.inf)
                if not np.isfinite(move_time):
                    continue

            candidate = (move_time, loc)

            if loc_name in banned_locations:
                fallback.append(candidate)
            else:
                preferred.append(candidate)

        if len(preferred) > 0:
            return sorted(preferred, key=lambda x: x[0])[0][1]

        if allow_banned_if_no_other and len(fallback) > 0:
            return sorted(fallback, key=lambda x: x[0])[0][1]

        return None

    def move_ap_direct(self, LN, destination, date=None, record_trace=True):
        self.last_move_error = None

        if LN not in self.APs:
            self.last_move_error = f'LN {LN} not in self.APs'
            return False, 0

        AP = self.APs[LN]
        origin = AP.Location

        if destination is None:
            self.last_move_error = f'LN {LN} destination is None'
            return False, 0

        if not destination.canPlace(AP):
            existing = destination.AP.get_LN() if destination.AP is not None else None
            self.last_move_error = (
                f'LN {LN} cannot place at {destination.name}; existing occupant={existing}'
            )
            return False, 0

        if origin is None:
            move_time = 0
        else:
            move_time = origin.time_to.get(destination.name, np.inf)

            if not np.isfinite(move_time):
                self.last_move_error = (
                    f'LN {LN} has no finite move time from {origin.name} to {destination.name}'
                )
                return False, 0

        if origin is not None:
            AP.path.append(origin.name)
            origin.unassign(date=date)
            self.chickenTracks[origin.name] = None

        assigned = destination.assign(AP, date=date)

        if not assigned:
            if origin is not None:
                origin.assign(AP, date=date)
                self.chickenTracks[origin.name] = LN

            self.last_move_error = f'LN {LN} assignment to {destination.name} failed'
            return False, 0

        AP.Location = destination
        self.chickenTracks[destination.name] = LN

        origin_name = None if origin is None else origin.name
        destination_name = destination.name

        if origin_name in ['BSC1', 'BSC2'] and destination_name not in ['BSC1', 'BSC2']:
            AP.taskCompletion['paint'] = True
            self.remove_from_queue('FGI task', LN, subqueue='paint')

        if destination_name == 'CR3' and AP.taskState == 'compass':
            AP.compassStartDate = date

        if record_trace and hasattr(self, 'trace'):
            self.trace.record_move(date, LN, destination.name)

        return True, move_time

    def move_ap_with_centerline(self, LN, destination, date=None):
        if LN not in self.APs:
            self.last_move_error = f'LN {LN} not in self.APs'
            return False, 0

        blockers = [
            blocker for blocker in self.get_centerline_blockers(destination)
            if blocker['LN'] != LN
        ]

        if len(blockers) == 0:
            return self.move_ap(LN, destination, date=date)

        total_move_time = 0
        staged = []

        for blocker in blockers:
            blocker_LN = blocker['LN']
            original_loc = blocker['loc']
            temp_loc = self.find_temp_location(original_loc)

            if temp_loc is None:
                self.last_move_error = f'No available temporary location for blocker LN {blocker_LN}'
                return False, 0

            moved, move_time = self.move_ap_direct(
                blocker_LN,
                temp_loc,
                date=date,
                record_trace=False
            )

            if not moved:
                for staged_item in reversed(staged):
                    self.move_ap_direct(
                        staged_item['LN'],
                        staged_item['original_loc'],
                        date=date,
                        record_trace=False
                    )

                return False, 0

            staged.append({
                'LN': blocker_LN,
                'original_loc': original_loc,
                'temp_loc': temp_loc
            })

            total_move_time += move_time

        moved_target, target_move_time = self.move_ap_direct(
            LN,
            destination,
            date=date,
            record_trace=True
        )

        if not moved_target:
            for staged_item in reversed(staged):
                self.move_ap_direct(
                    staged_item['LN'],
                    staged_item['original_loc'],
                    date=date,
                    record_trace=False
                )

            return False, 0

        total_move_time += target_move_time

        for staged_item in reversed(staged):
            moved_back, back_move_time = self.move_ap_direct(
                staged_item['LN'],
                staged_item['original_loc'],
                date=date,
                record_trace=False
            )

            if not moved_back:
                self.last_move_error = (
                    f"Could not return blocker LN {staged_item['LN']} "
                    f"to {staged_item['original_loc'].name}"
                )
                return False, total_move_time

            total_move_time += back_move_time

        AP = self.APs[LN]
        AP.moveReq = False
        AP.destination = None

        self.remove_from_queue('move', LN)

        self.chickenTracks[destination.name] = LN

        if CODECELL_OUTPUT:
            header('CENTERLINE MOVE EXECUTED')
            print(f'LN {LN} moved to {destination.name}; blockers temporarily staged and returned')
            line()

        return True, total_move_time

    def complete_AP(self, LN):
        if LN not in self.APs:
            return False

        AP = self.APs[LN]
        AP.taskState = 'delivered'

        self.remove_from_all_queues(LN)

        if AP.Location is not None:
            loc_name = AP.Location.name
            AP.Location.unassign()
            self.chickenTracks[loc_name] = None

        AP.Location = None
        self.APs.pop(LN)

        if CODECELL_OUTPUT:
            header('AP DELIVERED')
            print(f'LN {LN} has been delivered and removed from FGI')
            line()

        return True
    
    def build_daily_move_candidates(self):
        for LN, AP in self.APs.items():
            if AP.isMoveReq() and LN not in self.queues['move']:
                self.queues['move'].append(LN)
    
    def complete_ready_APs(self):
        exit_queue = []

        for loc_name, loc in self.Locations.items():
            ap = loc.AP

            if ap is None:
                continue

            LN = ap.get_LN()

            if LN not in self.APs:
                continue

            if not self.is_in_any_queue(LN):
                if LN not in exit_queue:
                    exit_queue.append(LN)

        for LN in exit_queue:
            self.complete_AP(LN)

        if CODECELL_OUTPUT and len(exit_queue) > 0:
            header('EXIT / DELIVERY CHECK')
            print(f'Delivered APs: {exit_queue}')
            line()

        return exit_queue

    def schedule_paint_moves(self, today, paint_schedule):
        tomorrow = today + pd.Timedelta(days=1)
        scheduled_moves = []
        queued_out = []
        queued_in = []

        if tomorrow not in paint_schedule:
            return scheduled_moves

        for bay, scheduled in paint_schedule[tomorrow].items():
            if bay not in self.Locations:
                continue

            actual_occupant_ap = self.Locations[bay].AP
            actual_occupant = actual_occupant_ap.get_LN() if actual_occupant_ap is not None else None

            if scheduled is None:
                if actual_occupant is not None and actual_occupant in self.APs:
                    self.APs[actual_occupant].requireMove()
                    self.APs[actual_occupant].taskState = 'btg_completion'

                    if actual_occupant not in self.queues['move']:
                        self.queues['move'].append(actual_occupant)

                    scheduled_moves.append(actual_occupant)
                    queued_out.append(actual_occupant)
                continue

            if scheduled not in self.APs:
                if CODECELL_OUTPUT:
                    header('PAINT MOVE ERROR', '~')
                    print(f'AP {scheduled} scheduled for paint at {bay}, but not found in FGI')
                    line('~')
                continue

            # tomorrow bay is physically occupied by different AP
            if actual_occupant is not None and actual_occupant != scheduled:
                if actual_occupant in self.APs:
                    self.APs[actual_occupant].requireMove()
                    self.APs[actual_occupant].taskState = 'btg_completion'

                    if actual_occupant not in self.queues['move']:
                        self.queues['move'].append(actual_occupant)

                    scheduled_moves.append(actual_occupant)
                    queued_out.append(actual_occupant)

            # scheduled AP moves into paint bay
            self.APs[scheduled].requireMove(destination=bay)
            self.APs[scheduled].taskState = 'paint'

            if scheduled not in self.queues['move']:
                self.queues['move'].append(scheduled)

            scheduled_moves.append(scheduled)
            queued_in.append(scheduled)

            if CODECELL_OUTPUT:
                header('AP added to move queue for paint assignment')
                if actual_occupant is not None and actual_occupant != scheduled:
                    print(f'AP {actual_occupant} and AP {scheduled} added to move queue for paint assignment at {bay}')
                else:
                    print(f'AP {scheduled} added to move queue for paint assignment at {bay}')
                line()

        if CODECELL_OUTPUT:
            header('PAINT STATUS')
            print(f"BSC1 occupant: {None if 'BSC1' not in self.Locations or self.Locations['BSC1'].AP is None else self.Locations['BSC1'].AP.get_LN()}")
            print(f"BSC2 occupant: {None if 'BSC2' not in self.Locations or self.Locations['BSC2'].AP is None else self.Locations['BSC2'].AP.get_LN()}")
            print(f"Paint queue length: {len(self.queues['FGI task']['paint'])}")
            print(f'APs queued to move out of BSC: {queued_out}')
            print(f'APs queued to move into BSC: {queued_in}')
            line()

        return scheduled_moves

    def schedule_compass_moves(self, today):
        scheduled_moves = []
        obstructions = []

        i = 0
        while i < len(self.queues['FGI task']['compass']):
            LN = self.queues['FGI task']['compass'][i]
            if LN not in self.APs:
                self.queues['FGI task']['compass'].pop(i)
                continue
            i += 1

        if 'CR3' not in self.Locations:
            return scheduled_moves, obstructions

        cr3 = self.Locations['CR3']
        cr1 = self.Locations.get('CR1')
        cr2 = self.Locations.get('CR2')

        if CODECELL_OUTPUT:
            header('COMPASS STATUS')
            print(f"CR3 occupant: {None if cr3.AP is None else cr3.AP.get_LN()}")
            print(f"CR1 occupant: {None if cr1 is None or cr1.AP is None else cr1.AP.get_LN()}")
            print(f"CR2 occupant: {None if cr2 is None or cr2.AP is None else cr2.AP.get_LN()}")
            print(f"Compass queue front: {None if len(self.queues['FGI task']['compass']) == 0 else self.queues['FGI task']['compass'][0]}")
            line()

        # STEP 1: handle current CR3 occupant before scheduling next compass AP
        if cr3.AP is not None:
            cr3_ap = cr3.AP
            cr3_LN = cr3_ap.get_LN()

            if cr3_LN not in self.APs:
                return scheduled_moves, obstructions

            cr1_clear = cr1 is None or cr1.AP is None
            cr2_clear = cr2 is None or cr2.AP is None

            has_start = hasattr(cr3_ap, 'compassStartDate') and cr3_ap.compassStartDate is not None
            has_waited_one_workday = has_start and pd.Timestamp(today) > pd.Timestamp(cr3_ap.compassStartDate)

            if cr1_clear and cr2_clear and has_waited_one_workday:
                cr3_ap.taskCompletion['compass'] = True
                cr3_ap.taskState = 'btg_completion'
                self.remove_from_queue('FGI task', cr3_LN, subqueue='compass')

                banned = []
                if len(self.queues['FGI task']['compass']) > 0:
                    banned = ['CR1', 'CR2']

                destination = self.get_closest_normal_destination(
                    cr3_ap,
                    banned_locations=banned,
                    allow_banned_if_no_other=True
                )

                if destination is not None:
                    cr3_ap.requireMove(destination=destination.name)

                    if cr3_LN not in self.queues['move']:
                        self.queues['move'].append(cr3_LN)

                    scheduled_moves.append(cr3_LN)

                if CODECELL_OUTPUT:
                    header('COMPASS COMPLETE')
                    print(f'AP {cr3_LN} completed compass at CR3')
                    print(f'CR1 clear: {cr1_clear}, CR2 clear: {cr2_clear}')
                    print(f'Move-away destination: {None if destination is None else destination.name}')
                    line()

                return scheduled_moves, obstructions

            else:
                if CODECELL_OUTPUT:
                    header('COMPASS WAITING')
                    print(f'AP {cr3_LN} remains in CR3')
                    print(f'CR1 clear: {cr1_clear}, CR2 clear: {cr2_clear}')
                    print(f'Has waited one workday: {has_waited_one_workday}')
                    line()

                return scheduled_moves, obstructions

        # STEP 2: CR3 is open, schedule next compass AP
        if len(self.queues['FGI task']['compass']) == 0:
            return scheduled_moves, obstructions

        LN = self.queues['FGI task']['compass'][0]

        if LN not in self.APs:
            self.remove_from_queue('FGI task', LN, subqueue='compass')
            return scheduled_moves, obstructions

        AP = self.APs[LN]
        AP.requireMove(destination='CR3')
        AP.taskState = 'compass'

        if LN not in self.queues['move']:
            self.queues['move'].append(LN)

        scheduled_moves.append(LN)

        if CODECELL_OUTPUT:
            header('AP added to move queue for compass calibration')
            print(f'AP {LN} added to move queue for compass calibration at CR3')
            line()

        return scheduled_moves, obstructions

    def process_move_queue(self, date=None, max_move_time=8, max_move_count=6):
        total_move_time = max_move_time
        move_count = 0
        blocked = []
        blocked_detail = {}

        i = 0

        while i < len(self.queues['move']) and total_move_time > 0 and move_count < max_move_count:
            LN = self.queues['move'][i]

            if LN not in self.APs:
                blocked_detail[LN] = 'missing_ap_removed_from_queue'
                self.queues['move'].pop(i)
                continue

            AP = self.APs[LN]

            if not AP.isMoveReq():
                blocked_detail[LN] = 'move_not_required_removed_from_queue'
                self.queues['move'].pop(i)
                continue

            if AP.destination is not None:
                destination = self.Locations.get(AP.destination)
                if destination is None:
                    blocked.append(LN)
                    blocked_detail[LN] = f'missing_destination_location: {AP.destination}'
                    i += 1
                    continue
            else:
                candidates = AP.get_move_candidates(self)
                if len(candidates) == 0:
                    if CODECELL_OUTPUT:
                        print(f'NO CANDIDATES for LN {LN} at {None if AP.Location is None else AP.Location.name}')
                    blocked.append(LN)
                    blocked_detail[LN] = 'no_candidates'
                    i += 1
                    continue
                destination = candidates[0]['destination']

            if destination is None:
                blocked.append(LN)
                blocked_detail[LN] = 'missing_destination'
                i += 1
                continue

            if not destination.canPlace(AP):
                existing = destination.AP.get_LN() if destination.AP is not None else None
                blocked.append(LN)
                blocked_detail[LN] = f'destination_unavailable: {destination.name}, occupant={existing}'
                i += 1
                continue

            moved, move_time = self.move_ap_with_centerline(LN, destination, date=date)

            if not moved:
                blocked.append(LN)
                blocked_detail[LN] = self.last_move_error if hasattr(self, 'last_move_error') else 'move_ap_failed'
                i += 1
                continue

            total_move_time -= move_time
            move_count += 1

            # do not increment i after successful move because move_ap removes LN from queue

        return {
            'move_count': move_count,
            'remaining_move_time': total_move_time,
            'blocked': blocked,
            'blocked_detail': blocked_detail,
            'remaining_queue': self.queues['move'].copy()
        }

    def record_day(self, date):
        self.schedule[date] = {}

        for loc_name, loc in self.Locations.items():
            self.schedule[date][loc_name] = None if loc.AP is None else loc.AP.get_LN()

        for LN, AP in self.APs.items():
            self.apStateRows.append({
                'Date': date,
                'LN': LN,
                'Location': None if AP.Location is None else AP.Location.name,
                'FGI_tot': AP.get_fgi_btg('FGI_tot'),
                'structure': AP.get_fgi_btg('structure'),
                'systems': AP.get_fgi_btg('systems'),
                'declam': AP.get_fgi_btg('declam'),
                'test': AP.get_fgi_btg('test'),
                'moveReq': AP.isMoveReq()
            })


In [10]:
class FGITrace:
    def __init__(self):
        self.chickentracks = {}
        self.labor_allocation = {}
        self.moves = {}
        self.btg_completion = {}

    def record_day_start(self, date, fgi):
        date = pd.Timestamp(date)

        self.chickentracks[date] = {
            loc_name: loc.AP.get_LN() if loc.AP is not None else None
            for loc_name, loc in fgi.Locations.items()
        }

    def record_labor(self, date, team, LNs):
        date = pd.Timestamp(date)

        if date not in self.labor_allocation:
            self.labor_allocation[date] = {}

        if isinstance(LNs, list):
            self.labor_allocation[date][team] = ', '.join(map(str, LNs))
        elif LNs is None:
            self.labor_allocation[date][team] = None
        else:
            self.labor_allocation[date][team] = str(LNs)

    def record_move(self, date, LN, target_location):
        date = pd.Timestamp(date)

        if date not in self.moves:
            self.moves[date] = {}

        self.moves[date][LN] = target_location

    def record_btg(self, date, LN, skill, btg_completed):
        date = pd.Timestamp(date)

        if skill not in self.btg_completion:
            self.btg_completion[skill] = {}

        if date not in self.btg_completion[skill]:
            self.btg_completion[skill][date] = {}

        current = self.btg_completion[skill][date].get(LN, 0)
        self.btg_completion[skill][date][LN] = current + btg_completed

    def to_dataframes(self):
        chickentracks_df = pd.DataFrame.from_dict(self.chickentracks, orient='index')
        chickentracks_df.index.name = 'Date'

        labor_df = pd.DataFrame.from_dict(self.labor_allocation, orient='index')
        labor_df.index.name = 'Date'

        moves_df = pd.DataFrame.from_dict(self.moves, orient='index')
        moves_df.index.name = 'Date'

        btg_dfs = {}
        for skill, data in self.btg_completion.items():
            df = pd.DataFrame.from_dict(data, orient='index')
            df.index.name = 'Date'
            btg_dfs[skill] = df

        return chickentracks_df, labor_df, moves_df, btg_dfs

<h4>b) <i>Functions</i>: Initialize AP

In [11]:
def init_APs(ap_df):
    return {
        str(row['LN']): AP(
            LN=int(row['LN']),
            faro=row['FA RO'],
            toB1R=row['FA RO to B1R'],
            counters=row['Total Counters'],
            btg={
                'tot': row['BTG_tot'],
                'p0': row['BTG_p0'],
                'p1': row['BTG_p1'],
                'p2': row['BTG_p2'],
                'p3': row['BTG_p3'],
                'engines': row['BTG_engines'],
                'doors': row['BTG_doors'],
                'test': row['BTG_test']
            },
            tankClosed=row['TankStatus'],
            ceilings=row['Ceilings'],
            testsRun=row['Initial Tests Run'],
            p3_milestones={
                'Engine Hang': row['P3_Engine Hang'],
                'Flight Controls': row['P3_Flight Controls'],
                'Gear Swing': row['P3_Gear Swing'],
                'Medium Pressure Test': row['P3_Medium Pressure Test'],
                'Oil On': row['P3_Oil On'],
                'Power On': row['P3_Power On'],
                'Engine_Type': row['P3_Engine_Type'],
                'Milestone_Completion_Percentage': row['P3_Milestone_Completion_Percentage']
            },
            shakes={'complete': row['shakes_complete'], 'req': row['shakes_req']}
        )
        for _, row in ap_df.iterrows()
    }

def init_Locations(location_df):
    Locations = {}
    for _, row in location_df.iterrows():
        if not pd.isna(row['Location']):
            Locations[row['Location']] = Location(
                priority=row['Future State Priority'],
                dateOnline=row['Date Online'],
                name=row['Location'],
                owner=row['Owner'],
                tooling={
                    'jacking': row['tooling_jacking'],
                    'wings': row['tooling_wings'],
                    'tankClosure': row['tooling_tankClosure']
                },
                centerlines = row['centerline_dependencies']
            )
    return Locations

<h3>Main Loop — Schedule Generation</h3>

In [12]:
# Initialize based on BSC input
APs = init_APs(ap_df)
Locations = init_Locations(location_df)


In [13]:
CODDECELL_OUTPUT = False

In [14]:
## MAIN ALGORITHM
# ============================= INITIALIZATION =================================
# scheduling/loop variables
START = pd.to_datetime(STARTDATE)
END = pd.to_datetime(ENDDATE)
dates = pd.date_range(START, END)

today = START
TERMINATION = False


for LN, ap in APs.items():
    ap.Location = None
    ap.moveReq = False
    ap.destination = None
    ap.path = []
    ap.schedule = {}
    ap.taskState = 'FA'

for loc_name, loc in Locations.items():
    loc.AP = None
    loc.schedule = {}


# initialize FGI object and add APs and Locations
fgi = FGI(labor=FGI_STAFFING_BYSHIFT, CPJ=FGI_CPJ)
trace = FGITrace()
fgi.trace = trace

for loc_name, loc in Locations.items():
    fgi.add_Location(loc_name, loc)


add_move_times(fgi, move_times)

paint_schedule = load_paint_schedule(filepath = rootpath+filepaths['Paint Schedule'], sheet_name='Historical')
# ============================= MAIN SCHEDULING LOOP ============================
while TERMINATION == False:
    if CODECELL_OUTPUT:
        header(f'{today.strftime("%Y-%m-%d")}', '=')
    if today > END:
        TERMINATION = True
        if CODECELL_OUTPUT: 
            header('END DATE REACHED', '>')
        if not FORECAST_UNTIL_COMPLETION:
            TERMINATION = True
            header('TERMINATION', '==>')
        else:
            header('FORECASTING MODE','>')
        line('>')
        break

    trace.record_day_start(today, fgi)
    # ————————| ROLLOUT APs FROM FA | —————————————————————————————————————————————————
    for LN, ap in APs.items():
        if LN not in fgi.APs and today == ap.get_FAROdate():
            fgi.add_ap(ap)

            if CODECELL_OUTPUT:
                header('RO')
                print(f'AP {LN} rolled out from FA on {today.strftime("%Y-%m-%d")}')
                print('Added to FGI object')
                line()


    shift = 0
    while shift < 3 and len(fgi.APs) > 0:
        # ————————| STATUS UPDATES | —————————————————————————————————————————————————
        if shift == 0:
            for LN, AP in fgi.APs.items():
                AP.toB1R -= 1

            #fgi.prioritize_queues[QUEUING_RULE]

            if today.weekday() < 5:
                if CODECELL_OUTPUT:
                    print('Workday')

            else:
                shift = 4
            
            pass # -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   -   
        
        # ————————| LABOR COMPLETION | —————————————————————————————————————————————————
        else:
            if CODECELL_OUTPUT:
                header(f'Shift: {shift}')
                header('Labor allocation', '-')
            for team in FGI_TEAMS:

                btg_remaining = fgi.get_btg_completion(team, shift)
                labor_queue = fgi.queues['labor'][team]

                if CODECELL_OUTPUT:
                    print(f'\n{team.capitalize()} Team\nBTG: {btg_remaining} | Queue: {labor_queue}')
                
                worked_lns, btg_remaining = fgi.assign_labor(team, max_btg_completion=btg_remaining, date=today)
                trace.record_labor(today, team, worked_lns)
                if btg_remaining > 0:
                    if CODECELL_OUTPUT:
                        print(f"Idle Time for {team}, with {btg_remaining} possible BTG completions")


        # fgi.assign_labor(shift)
        shift += 1
    
    # —————————————— MOVE APs (3rd shift, non-buffer)——————————————————————————————————————————————
    if shift == 3 and today not in PLANNED_BUFFER_DAYS:
        MAX_MOVE_COUNT = 6

        obstructions = []
        centerlines = []

        paint_moves = fgi.schedule_paint_moves(today, paint_schedule)
        compass_moves, obstructions = fgi.schedule_compass_moves(today)

        fgi.build_daily_move_candidates()

        move_summary = fgi.process_move_queue(
            date=today,
            max_move_time=8,
            max_move_count=MAX_MOVE_COUNT
        )

        if CODECELL_OUTPUT:
            header('MOVE SUMMARY', '-')
            print(move_summary)
            line('-')
        # —————————————— AP DELIVERY——————————————————————————————————————————————
        

        # ——————————————| EXIT / DELIVERY CHECK |—————————————————————————————————————————————— 
        exit_queue = fgi.complete_ready_APs()

    today += pd.Timedelta(days=1)
    # break


————————————————————————————————————————————————————————————————————————————————————————————————————

Location D1 added to FGI
Current locations in FGI: ['D1']
————————————————————————————————————————————————————————————————————————————————————————————————————

————————————————————————————————————————————————————————————————————————————————————————————————————

Location D2 added to FGI
Current locations in FGI: ['D1', 'D2']
————————————————————————————————————————————————————————————————————————————————————————————————————

————————————————————————————————————————————————————————————————————————————————————————————————————

Location A1 added to FGI
Current locations in FGI: ['D1', 'D2', 'A1']
————————————————————————————————————————————————————————————————————————————————————————————————————

————————————————————————————————————————————————————————————————————————————————————————————————————

Location A2 added to FGI
Current locations in FGI: ['D1', 'D2', 'A1', 'A2']
——————————————————

In [17]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

output_file = 'scheduler_trace_output.xlsx'

chickentracks_df, labor_df, moves_df, btg_dfs = trace.to_dataframes()

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    chickentracks_df.to_excel(writer, sheet_name='ChickenTracks')
    labor_df.to_excel(writer, sheet_name='Labor Allocation')
    moves_df.to_excel(writer, sheet_name='Moves Per Day')

    for skill, df in btg_dfs.items():
        sheet_name = f'BTG {skill}'[:31]
        df.to_excel(writer, sheet_name=sheet_name)

wb = load_workbook(output_file)

header_fill = PatternFill('solid', fgColor='1F4E78')
header_font = Font(color='FFFFFF', bold=True)
index_fill = PatternFill('solid', fgColor='D9EAF7')
thin_gray = Side(style='thin', color='BFBFBF')

for ws in wb.worksheets:
    ws.freeze_panes = 'B2'
    ws.sheet_view.showGridLines = False

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center')

    for cell in ws['A']:
        cell.fill = index_fill
        cell.font = Font(bold=True)
        cell.number_format = 'yyyy-mm-dd'
        cell.alignment = Alignment(horizontal='center')

    for row in ws.iter_rows():
        for cell in row:
            cell.border = Border(bottom=thin_gray)
            cell.alignment = Alignment(vertical='center', wrap_text=True)

    for col_idx, col_cells in enumerate(ws.columns, start=1):
        max_len = 0
        col_letter = get_column_letter(col_idx)

        for cell in col_cells:
            value = cell.value
            if value is not None:
                max_len = max(max_len, len(str(value)))

        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 30)

    ws.auto_filter.ref = ws.dimensions

wb.save(output_file)

print(f'Exported trace workbook: {output_file}')

Exported trace workbook: scheduler_trace_output.xlsx


In [18]:
trace_file = 'scheduler_trace_output.xlsx'

ct = pd.read_excel(trace_file, sheet_name='ChickenTracks')
moves = pd.read_excel(trace_file, sheet_name='Moves Per Day')

temp_cols = [
    c for c in ct.columns
    if str(c).startswith('N')
]

temp_occupied = ct[temp_cols].notna().sum().sum() if len(temp_cols) > 0 else 0

move_values = moves.drop(columns=[moves.columns[0]], errors='ignore').values.flatten()
temp_final_moves = sum(str(v).startswith('N') for v in move_values if pd.notna(v))

print('Temporary-location occupied cells in ChickenTracks:', temp_occupied)
print('Temporary-location final move targets in Moves Per Day:', temp_final_moves)

Temporary-location occupied cells in ChickenTracks: 0
Temporary-location final move targets in Moves Per Day: 0
